In [ ]:
%matplotlib inline
import pathlib as pl
import numpy as np
import sys
import xugrid
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import rasterio.warp
from shapely.geometry import shape

import scipy.sparse as sparse

import flopy
import flopy.plot.styles as styles

from gdptools import WeightGenP2P

In [ ]:
sys.path.append("../common")
from liss_settings import cx, cx_provider, extent, boxx, boxy, get_dflow_grid_name, get_modflow_coupling_tag, get_modflow_grid_name

In [ ]:
control_path = pl.Path("../dflow-fm/highres/tides_atm_surge_2018/FlowFM.mdu")# change this if using a different D-Flow FM control file
grid_name = get_dflow_grid_name(control_path)
print(grid_name)


In [ ]:
# mf_grid_name = get_modflow_grid_name()
# print(mf_grid_name)

In [ ]:
get_modflow_coupling_tag(1.)

## Read the D-Flow FM output file

Make sure you run D-Flow FM by itself first so that there is an output NetCDF file available so that the mapping is done using the internal node order

In [ ]:
# use an output file because this is what will be available from bmi and is in the correct order
source_path = r"../dflow-fm/highres/tides_atm_surge_2018/run_dflow_only/output/FlowFM_map.nc"
source_ds = xugrid.open_dataset(source_path)

### Convert the NetCDF data to a geodataframe

In [ ]:
source_gdf = source_ds["mesh2d_nFaces"].ugrid.to_geodataframe(name="cell")

In [ ]:
source_gdf.set_crs(32618, inplace=True)

In [ ]:
print(source_gdf.crs)

## Open the shapefile with the location of the coastal boundaries in MODFLOW

The shapefile needs to be limited to coastal boundary locations and be in the same coordinate system as the D-Flow FM model (UTM 18N).

In [ ]:
fpth = f"../modflow/gis/PJ/PJ_CHD_utm18n.shp"
print(fpth)

In [ ]:
# nc_1 = xugrid.open_dataset(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\dflow-fm\coarse\20250821\base\LIS_modflow_bathy2_net.nc")
# nc_1_gdf = gpd.GeoDataFrame(geometry = gpd.points_from_xy(nc_1.mesh2d_node_x , nc_1.mesh2d_node_y), crs = 'EPSG:32618')
# nc2 = xugrid.open_dataset(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\dflow-fm\coarse\tides_2018\base\LIS_GPT_PJ_cells3_net.nc")
# nc_2_gdf = gpd.GeoDataFrame(geometry = gpd.points_from_xy(nc2.mesh2d_node_x , nc2.mesh2d_node_y), crs = 'EPSG:32618')

# a = nc_1_gdf.explore()
# #nc_2_gdf.explore(color = 'red')
# a

In [ ]:
target_coastal = gpd.read_file(fpth) # this is the shapefile with coastal boundary conditions
a = target_coastal.explore()

dflowdf = gpd.GeoDataFrame(geometry = gpd.points_from_xy(source_ds.mesh2d_node_x , source_ds.mesh2d_node_y), crs = 'EPSG:32618')
dflowdf.explore(m=a)

In [ ]:
ax = target_coastal.plot(alpha=0.25, column="boundname", cmap= 'rainbow')
#cx.add_basemap(ax, crs=target_coastal.crs, attribution=False, source=cx_provider)

## Create the D-FLOW FM to CHD mapping

In [ ]:
# generate the weights
weight_gen = WeightGenP2P(
    target_poly=target_coastal,
    target_poly_idx="chd_no",
    source_poly=source_gdf,
    source_poly_idx=["cell"],
    method="serial",
    weight_gen_crs=32618,
)
weights = weight_gen.calculate_weights()


In [ ]:
weights[:12]
# len(weights)

In [ ]:
map_shape = (target_coastal.shape[0], source_gdf.shape[0])
map_shape

In [ ]:
dflow2mfchd = np.zeros(map_shape, dtype=float)
print(f"{dflow2mfchd.shape}\n{dflow2mfchd}")

In [ ]:
for r,c,v in zip(weights["chd_no"], weights["cell"], weights["wght"]):
    print(r,c,v)
    dflow2mfchd[int(float(r)),int(c)] = v

check = weights['chd_no'].drop_duplicates()
assert len(check) == target_coastal.shape[0]

## Create the chd masking array

Where the sums of the weights along a row are not equal to ~1.0

In [ ]:
mask_idx = np.isclose(dflow2mfchd.sum(axis=1), 1.0)
print(f"{mask_idx.sum()}\n{mask_idx.shape}\n{mask_idx}")


print(target_coastal.shape[0], mask_idx.shape[0])
assert mask_idx.shape[0] == target_coastal.shape[0]


# Print indices of rows where the condition is False
false_rows = np.where(~mask_idx)[0]
print("Rows where the sum is NOT close to 1.0:")
print(false_rows)

f = dflow2mfchd[false_rows]
f = dflow2mfchd[false_rows].sum()
f

### Test the D-FLOW FM to CHD mapping

In [ ]:
s = np.full(source_gdf.shape[0], 1.0)
h = np.full(mask_idx.shape, 2.0)
h[mask_idx] = dflow2mfchd.dot(s)[mask_idx]
s.shape, dflow2mfchd.shape, h.shape

In [ ]:
print(f"{h.sum()}\n{h}")

#### Test with a nan

In [ ]:
s = np.random.random(source_gdf.shape[0])
s[1544] = -1e30
print(s)

In [ ]:
h = np.full(mask_idx.shape, 2.0)
h = dflow2mfchd.dot(s)
h.shape

In [ ]:
print(f"{h.sum()}\n{h}")

## Create the CHD to Qext mapping

In [ ]:
chd2qext = np.transpose(dflow2mfchd.copy())

### Test the CHD to Qext mapping

In [ ]:
q = np.full(chd2qext.shape[1], 1.0)

In [ ]:
qext = chd2qext.dot(q)

In [ ]:
print(f"{qext.sum()}\n{qext.shape}")

## Save the mapping arrays

In [ ]:
print(target_coastal.shape[0], source_gdf.shape[0])
assert dflow2mfchd.shape == (target_coastal.shape[0], source_gdf.shape[0])

In [ ]:
print(mask_idx.shape[0])
assert mask_idx.shape[0] == target_coastal.shape[0]

In [ ]:
assert chd2qext.shape == (source_gdf.shape[0], target_coastal.shape[0])
print(source_gdf.shape[0], target_coastal.shape[0])

In [ ]:
mf_grid_name = get_modflow_grid_name()
fpath = f"../mapping/PJ/dflow{grid_name}_to_{mf_grid_name}_chd.npz"
np.savez_compressed(fpath, dflow2mfchd=dflow2mfchd, chdmask=mask_idx, chd2qext=chd2qext)

In [ ]:
mf_grid_name